# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a demonstration for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema JSON-LD URL)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Publication Date: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here we display the available record sets and for each one, list its fields and column `@id`s.

In [ ]:
# Examine record sets and their fields. All entities are referenced by their @id.
from pprint import pprint

record_sets = dataset.record_sets

if record_sets:
    print("Available record sets and their fields:")
    record_set_ids = []
    for rs in record_sets:
        print(f"\nRecord set @id: {rs.id}")
        record_set_ids.append(rs.id)
        # Each field in a record set
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id}, name: {getattr(field, 'name', '')}")
        # Columns/Files
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    Column @id: {col.id}, name: {getattr(col, 'name', '')}")
else:
    print("No record sets defined directly in the metadata. Attempting to infer from files...")
    record_set_ids = []    # Will be filled in next cell.

### Listing first 3 records from each record set

Let's inspect sample records for each available record set, referencing them by their `@id`.

In [ ]:
# Preview a few samples from every record set
if record_sets:
    for rs in record_sets:
        print(f"\nRecord set @id: {rs.id}")
        try:
            for idx, rec in enumerate(dataset.records(record_set=rs.id)):
                print(rec)
                if idx >= 2:
                    break
        except Exception as e:
            print(f"  Could not read records: {e}")
else:
    print("No record sets to preview records from.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract the data for **each record set** and load it into a pandas DataFrame using its `@id` as the key.

In [ ]:
# Build a dictionary of DataFrames indexed by record set @id
dataframes = {}

if record_sets:
    for rs in record_sets:
        record_set_id = rs.id
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    # Pick the first record set id for demonstration
    first_rs = record_sets[0].id
    print(f"Columns for record set '@id': {first_rs}")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No record sets present. Data could not be extracted into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping data. Here, we demonstrate this for one record set and numeric field.

In [ ]:
import numpy as np

if record_sets:
    # Use the first record set as example
    first_rs = record_sets[0].id
    df = dataframes[first_rs]
    # Try to identify a numeric field by type or column name heuristics
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try guessing from column names like 'log_likelihood', 'coef', etc.
        for col in df.columns:
            if any(x in col.lower() for x in ["log_likelihood", "ll", "coef", "std", "pvalue", "se"]):
                numeric_cols.append(col)
        # Convert to numeric if possible
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use the first found numeric column
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in record set '@id' {first_rs} with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group by a likely categorical field:
        categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
        group_field = None
        for col in categorical_cols:
            if col.lower() not in ["id", "index"] and df[col].nunique() > 1:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"Grouped data for filtered records by '{group_field}', mean {numeric_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("No record sets to perform EDA on.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below is an example histogram of a numeric field from the record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No suitable numeric field available for histogram visualization.")

## 6. Conclusion

In this notebook, we loaded a Croissant-described dataset using `mlcroissant`, explored its available record sets and fields (referenced by `@id`), loaded data into DataFrames by record set, and performed simple exploratory analysis and visualization on numeric columns. This approach supports transparent, reproducible data science using FAIR dataset standards.